# DAVE Documents API — Authentication & Shared Setup

This notebook handles **authentication** and stores the token in a shared variable
that is imported by all other notebooks.

The DAVE documents backend supports two auth modes:

| Mode | Description |
|------|-------------|
| **Local JWT** | `POST /api/auth/login` — email + password stored in MongoDB |
| **Keycloak** | `POST /api/auth/keycloak-login` — ROPC grant against Keycloak |
| **No auth** | `USE_AUTH=false` in `.env` — all requests pass through |

Run this notebook first; the others import `auth_state.py` for the token.

In [10]:
# ── dependencies ──────────────────────────────────────────────────────────────
# pip install requests ipywidgets  (run once)
import requests
import json
import os

In [11]:
# ── configuration ─────────────────────────────────────────────────────────────
# Adjust BASE_URL to wherever the documents service is running.
BASE_URL  = os.getenv("DAVE_API_URL", "http://localhost:3001")
API_BASE  = f"{BASE_URL}/api"

# Credentials — override with env vars or edit directly
EMAIL     = os.getenv("DAVE_EMAIL",    "viewer@dave.com")
PASSWORD  = os.getenv("DAVE_PASSWORD", "daveViewer42!")

# Set USE_AUTH to False if the server is running with USE_AUTH=false
USE_AUTH  = os.getenv("DAVE_USE_AUTH", "true").lower() != "false"

print(f"API base : {API_BASE}")
print(f"Auth     : {'enabled' if USE_AUTH else 'disabled (USE_AUTH=false)'}")

API base : http://localhost:3001/api
Auth     : enabled


## 1 — Login via Keycloak

`POST /api/auth/keycloak-login`

Use this block instead if your server is configured with Keycloak.

In [12]:
# ── Keycloak login (uncomment to use) ─────────────────────────────────────────
resp = requests.post(f"{API_BASE}/auth/keycloak-login", json={
    "username": EMAIL,
    "password": PASSWORD,
})
resp.raise_for_status()
kc_data       = resp.json()
access_token  = kc_data["access_token"]
refresh_token = kc_data["refresh_token"]
print("Keycloak login OK")

Keycloak login OK


## 2 — Persist state for other notebooks

Writes `auth_state.py` next to this notebook so the others can just
`from auth_state import ...`.

In [9]:
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
state_file   = os.path.join(notebook_dir, "auth_state.py")

with open(state_file, "w") as f:
    f.write(f'BASE_URL     = "{BASE_URL}"\n')
    f.write(f'API_BASE     = "{API_BASE}"\n')
    f.write(f'ACCESS_TOKEN = {repr(access_token)}\n')
    f.write(f'REFRESH_TOKEN = {repr(refresh_token)}\n')
    f.write(f'USE_AUTH     = {USE_AUTH}\n')
    f.write("""

def auth_headers():
    if ACCESS_TOKEN:
        return {"Authorization": f"Bearer {ACCESS_TOKEN}"}
    return {}
""")

print(f"auth_state.py written to: {state_file}")

auth_state.py written to: /Users/rubenagazzi/Documents/universita/DAVE/DAVE/notebooks/auth_state.py
